# 2.2 Workflow - Routing 路由

- **对应文档**: [workflow_routing](https://doc.agentscope.io/tutorial/workflow_routing.html)
- **本讲目标**: 掌握根据消息或条件将请求路由到不同 Agent 的用法。
- **前置知识**: 02_react_agent, 05_workflow_handoffs。

## 学习要点

- Routing 决定「当前请求由哪个 Agent 处理」。
- 可结合条件、关键词或 LLM 判断实现路由逻辑。

In [ ]:
# 在此按 doc 编写 routing 示例
# 参考 https://doc.agentscope.io/tutorial/workflow_routing.html
print("请参考 workflow_routing 文档完成 Routing 示例代码")

1. 使用结构化输出的显式 routing

In [1]:
import asyncio
import json
import os
from typing import Literal

from pydantic import BaseModel, Field

from agentscope.agent import ReActAgent
from agentscope.formatter import DashScopeChatFormatter
from agentscope.memory import InMemoryMemory
from agentscope.message import Msg
from agentscope.model import DashScopeChatModel
from agentscope.tool import Toolkit, ToolResponse

router = ReActAgent(
    name="Router",
    sys_prompt="你是一个路由智能体。你的目标是将用户查询路由到正确的后续任务，注意你不需要回答用户的问题。",
    model=DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=False,
    ),
    formatter=DashScopeChatFormatter(),
)

In [2]:
# 使用结构化输出指定路由任务
class RoutingChoice(BaseModel):
    your_choice: Literal[
        "Content Generation",
        "Programming",
        "Information Retrieval",
        None,
    ] = Field(
        description="选择正确的后续任务，如果任务太简单或没有合适的任务，则选择 ``None``",
    )
    task_description: str | None = Field(
        description="任务描述",
        default=None,
    )

In [3]:
async def example_router_explicit() -> None:
    """使用结构化输出进行显式路由的示例。"""
    msg_user = Msg(
        "user",
        "帮我写一首诗",
        "user",
    )

    # 路由查询
    msg_res = await router(
        msg_user,
        structured_model=RoutingChoice,
    )

    # 结构化输出存储在 metadata 字段中
    print("结构化输出：")
    print(json.dumps(msg_res.metadata, indent=4, ensure_ascii=False))

In [4]:
await example_router_explicit()

D:\PycharmProjects\agentscope-playground\.venv\Lib\site-packages\agentscope\model\_dashscope_model.py:231: DeprecationWarning: 'required' is not supported by DashScope API. It will be converted to 'auto'.
  warnings.warn(


Router: {
    "type": "tool_use",
    "name": "generate_response",
    "input": {
        "your_choice": "Content Generation",
        "task_description": "用户请求创作一首诗"
    },
    "id": "call_717b2f1695b54058bd35b8"
}
system: {
    "type": "tool_result",
    "id": "call_717b2f1695b54058bd35b8",
    "name": "generate_response",
    "output": [
        {
            "type": "text",
            "text": "Successfully generated response."
        }
    ]
}
Router: 我已经将您的请求路由到了诗歌创作任务。接下来，您将会得到一首为您特别创作的诗。请稍等片刻。
结构化输出：
{
    "your_choice": "Content Generation",
    "task_description": "用户请求创作一首诗"
}


2. 使用工具调用的隐式 routing, 主agent会自动路由并使用子agent封装的tool

In [5]:
async def generate_python(demand: str) -> ToolResponse:
    """根据需求生成 Python 代码。

    Args:
        demand (``str``):
            对 Python 代码的需求。
    """
    # 示例需求智能体
    python_agent = ReActAgent(
        name="PythonAgent",
        sys_prompt="你是一个 Python 专家，你的目标是根据需求生成 Python 代码。",
        model=DashScopeChatModel(
            model_name="qwen-max",
            api_key=os.environ["DASHSCOPE_API_KEY"],
            stream=False,
        ),
        memory=InMemoryMemory(),
        formatter=DashScopeChatFormatter(),
        toolkit=Toolkit(),
    )
    msg_res = await python_agent(Msg("user", demand, "user"))

    return ToolResponse(
        content=msg_res.get_content_blocks("text"),
    )

In [6]:
# 为演示目的模拟一些其他工具函数
async def generate_poem(demand: str) -> ToolResponse:
    """根据需求生成诗歌。

    Args:
        demand (``str``):
            对诗歌的需求。
    """
    pass

In [7]:
async def web_search(query: str) -> ToolResponse:
    """在网络上搜索查询。

    Args:
        query (``str``):
            要搜索的查询。
    """
    pass

In [8]:
toolkit = Toolkit()
toolkit.register_tool_function(generate_python)
toolkit.register_tool_function(generate_poem)
toolkit.register_tool_function(web_search)

# 使用工具模块初始化路由智能体
router_implicit = ReActAgent(
    name="Router",
    sys_prompt="你是一个路由智能体。你的目标是将用户查询路由到正确的后续任务。",
    model=DashScopeChatModel(
        model_name="qwen-max",
        api_key=os.environ["DASHSCOPE_API_KEY"],
        stream=False,
    ),
    formatter=DashScopeChatFormatter(),
    toolkit=toolkit,
    memory=InMemoryMemory(),
)


async def example_router_implicit() -> None:
    """使用工具调用进行隐式路由的示例。"""
    msg_user = Msg(
        "user",
        "帮我在 Python 中生成一个快速排序函数",
        "user",
    )

    # 路由查询
    await router_implicit(msg_user)


await example_router_implicit()

Router: {
    "type": "tool_use",
    "name": "generate_python",
    "input": {
        "demand": "生成一个快速排序函数"
    },
    "id": "call_63a2b1a985fa4e4abae942"
}
PythonAgent: 当然！快速排序是一种高效的排序算法，它使用分治法策略来递归地将数组分成更小的部分进行排序。以下是一个用 Python 实现的快速排序函数：

```python
def quicksort(arr):
    if len(arr) <= 1:
        return arr
    else:
        pivot = arr[len(arr) // 2]
        left = [x for x in arr if x < pivot]
        middle = [x for x in arr if x == pivot]
        right = [x for x in arr if x > pivot]
        return quicksort(left) + middle + quicksort(right)

# 示例用法
arr = [3, 6, 8, 10, 1, 2, 1]
sorted_arr = quicksort(arr)
print(sorted_arr)
```

这个实现使用了列表推导式来创建三个子数组：`left`（包含所有小于基准值的元素）、`middle`（包含所有等于基准值的元素）和 `right`（包含所有大于基准值的元素）。然后递归地对 `left` 和 `right` 子数组进行排序，并将结果合并。

如果你需要一个更传统的原地排序版本，可以参考以下实现：

```python
def quicksort_inplace(arr, low, high):
    if low < high:
        pi = partition(arr, low, high)
        quicksort_inplace(arr, low, pi - 1)
        quicksort_inplace(arr, pi + 1, high)
